<a href="https://colab.research.google.com/github/toobajaved/GraphBasedMovieRecommendation/blob/main/03_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

base = '/content/drive/MyDrive/MLG Project/Project'

BASE = '/content/drive/MyDrive/MLG Project/Project'
RAW_DIR = f'{BASE}/data/raw'
OUT_DIR = f'{BASE}/data/processed'
FIGURES = f'{BASE}/results/figures'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

# ---------- config ----------
POS_THRESHOLD = 4.0
TRAIN_FRAC = 0.70
VAL_FRAC = 0.10
TEST_FRAC = 0.20

NEG_PER_POS = 50  # use 20/50 for progress report speed; later you can do 99
SEED = 42
rng = np.random.default_rng(SEED)

# ---------- load ----------
ratings = pd.read_csv(f'{RAW_DIR}/ratings.csv')  # userId,movieId,rating,timestamp
movies = pd.read_csv(f'{RAW_DIR}/movies.csv')    # movieId,title,genres

# ---------- positives ----------
pos = ratings.loc[ratings["rating"] >= POS_THRESHOLD, ["userId", "movieId", "timestamp"]].copy()

# keep latest rating per (userId, movieId) if duplicates exist
pos = pos.sort_values("timestamp").drop_duplicates(["userId", "movieId"], keep="last")

# ---------- chronological split ----------
pos = pos.sort_values("timestamp").reset_index(drop=True)
n = len(pos)

train_end = int(TRAIN_FRAC * n)
val_end = int((TRAIN_FRAC + VAL_FRAC) * n)

train_pos = pos.iloc[:train_end].copy()
val_pos = pos.iloc[train_end:val_end].copy()
test_pos = pos.iloc[val_end:].copy()

# ---------- cold start prevention ----------
train_users = set(train_pos["userId"])
train_movies = set(train_pos["movieId"])

def move_cold_start_to_train(split_df, train_df):
    """Move interactions involving unseen users/movies into train."""
    global train_users, train_movies
    mask_ok = split_df["userId"].isin(train_users) & split_df["movieId"].isin(train_movies)
    ok = split_df.loc[mask_ok].copy()
    cold = split_df.loc[~mask_ok].copy()
    if len(cold) > 0:
        train_df = pd.concat([train_df, cold], ignore_index=True).sort_values("timestamp")
        train_users = set(train_df["userId"])
        train_movies = set(train_df["movieId"])
    return ok, train_df

val_pos, train_pos = move_cold_start_to_train(val_pos, train_pos)
test_pos, train_pos = move_cold_start_to_train(test_pos, train_pos)

# refresh sets
train_users = set(train_pos["userId"])
train_movies = set(train_pos["movieId"])

# user -> set of positive movies (use all positives so negatives are truly unseen)
user_pos_all = pos.groupby("userId")["movieId"].apply(set).to_dict()

train_movie_list = np.array(sorted(list(train_movies)))

# ---------- negative sampling ----------
def sample_negatives_for_split(split_pos_df, split_name):
    rows = []
    for u, m_pos in zip(split_pos_df["userId"].values, split_pos_df["movieId"].values):
        seen = user_pos_all.get(u, set())
        negs = []
        # sample until enough negatives
        while len(negs) < NEG_PER_POS:
            m_neg = int(rng.choice(train_movie_list))
            if m_neg != m_pos and m_neg not in seen:
                negs.append(m_neg)

        # store one row per candidate movie
        rows.append((u, m_pos, 1))
        for m_neg in negs:
            rows.append((u, m_neg, 0))

    out = pd.DataFrame(rows, columns=["userId", "movieId", "label"])
    out["split"] = split_name
    return out

val_candidates = sample_negatives_for_split(val_pos, "val")
test_candidates = sample_negatives_for_split(test_pos, "test")

# ---------- save ----------
train_pos.to_csv(f'{OUT_DIR}/train_pos.csv', index=False)
val_pos.to_csv(f'{OUT_DIR}/val_pos.csv', index=False)
test_pos.to_csv(f'{OUT_DIR}/test_pos.csv', index=False)

val_candidates.to_csv(f'{OUT_DIR}/val_candidates.csv', index=False)
test_candidates.to_csv(f'{OUT_DIR}/test_candidates.csv', index=False)

# ---------- quick stats for report ----------
stats = {
    "total_positive_edges": len(pos),
    "train_pos_edges": len(train_pos),
    "val_pos_edges": len(val_pos),
    "test_pos_edges": len(test_pos),
    "train_unique_users": train_pos["userId"].nunique(),
    "train_unique_movies": train_pos["movieId"].nunique(),
    "val_candidates_rows": len(val_candidates),
    "test_candidates_rows": len(test_candidates),
}
print(stats)

with open(f'{OUT_DIR}/split_stats.txt', "w") as f:
    for k, v in stats.items():
        f.write(f"{k}: {v}\n")

{'total_positive_edges': 48580, 'train_pos_edges': 47746, 'val_pos_edges': 378, 'test_pos_edges': 456, 'train_unique_users': 609, 'train_unique_movies': 6298, 'val_candidates_rows': 19278, 'test_candidates_rows': 23256}
